# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [ ]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [ ]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [ ]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Если клиентам подписки дать кешбек 5% на аптеки, то средний оборот в партнерских сервисах вырастет не меньше, чем на 200 рублей за 4 неедели, потому что это замотивирует потребителя на доп покупки |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Так как клиент из теста и контроля могут делить одну подписку, то получим протечку эффекта акции. Поэтому стоит выбрать подписку, но получим меньшее единиц наблюдения с большим шумом |
| группы и доли | control и test, делим 50/50, по ~3 000 клиентов в каждой, клиенты одной подписки попадают в одну группу |
| основная метрика | spend: оборот в партнёрских сервисах за 4 недели, ₽ |
| метрики-стражи: что не должно ухудшиться | Отток: увеличение количества оттока не больше чем на 0,2 п.п.; обращения: рост не больше чем на 0,01 обращения на клиента |
| α, мощность, MDE — числами из ячейки выше | α = 0,05, мощность 80 %, одно сравнение, 3 000 на группу; 200 ₽ выше MDE с CUPED (187), но ниже MDE без CUPED (236), поэтому CUPED нужен, чтобы вообще заметить такой эффект.|
| поправка на множественные сравнения | Используем поправку Холма, сравниваем четыре метрики (оборот, покупки, отток, обращения) |
| оценщик | разность средних с CUPED, ковариата spend_prev (измерена до эксперимента) |
| окно | 4 недели |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Запускаем, если нижняя граница интервала эффекта (с CUPED) выше 100(около +3% к среднему обороту) и ни один страж не ухудшился; дорабатываем и повторяем, если не попало ни в запуск, ни в отказ; отказываемся, если верхняя граница интервала ниже 100.|

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [ ]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [ ]:
print("размеры:", sizes.to_dict(), "· p SRM:", round(srm.pvalue, 3))
print("spend_prev до акции:", round(test["spend_prev"].mean(), 2), round(control["spend_prev"].mean(), 2),
      "· p:", round(balance.pvalue, 3))
print("эффект:", round(effect["diff"], 2), round(effect["lo"], 2), round(effect["hi"], 2), "· p:", round(effect["p"], 3))
print("MDE:", round(mde_raw), "· MDE с CUPED:", round(mde_cuped), "· +3 %:", round(three_pct))
print("p t / u:", ab["TTest p-value"].iloc[0], ab["UTest p-value"].iloc[0])

#### ✍️ Ответ на вопрос 1

Выборку получилось разбить хорошо: SRM p = 0,938 и баланс по spend_prev p = 0,372: оба выше 0,05. Эффект в то же время оказался малым по сравнению с MDE: 24,22 рубля с доверительным интервалом в (-144,4; 192,85) против 236 рублей MDE Без CUPED и 187 рублей с CUPED. Эффект в +3% из семинара 7 значительно меньше обоих MDE. Интервал проходит через ноль, поэтому нельзя сказать, что эффект доказан. С другой стороны, нельзя сказать, что эффекта нет, так как интервал вмещает и 97, и 190 рублей. 97 ₽ меньше обоих MDE (187 и 236), значит, эксперимент такого размера не рассчитан на такой эффект. Он мог пропустить реальный рост, и это объясняет широкий интервал. t-критерий (0,778) и U-критерий (0,465) оба больше 0,05, поэтому оба не находят различий в выборках.

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [ ]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [ ]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [ ]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [ ]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [ ]:
w_raw = effect["hi"] - effect["lo"]
w_cuped = effect_cuped["hi"] - effect_cuped["lo"]
print("средний spend_prev: test", round(test["spend_prev"].mean(), 2), "control", round(control["spend_prev"].mean(), 2))
print("ширина интервала:", round(w_raw, 2), "->", round(w_cuped, 2), "· сужение", round((1 - w_cuped / w_raw) * 100, 1), "%")
print("оценка:", round(effect["diff"], 2), "->", round(effect_cuped["diff"], 2))
print("p:", round(effect["p"], 4), "->", round(effect_cuped["p"], 4))
print("метрики после Холма:\n", metrics[["feature", "TTest p-value", "holm"]].round(3).to_string(index=False))

#### ✍️ Ответ на вопрос 2

Защитные метрики не сработали по оттоку: -0,41 п.п., p = 0,405; обращения незначимо выросли: +0,003, p = 0,663. Поправка Холма не оставила значимых признаков, минимальное p = 0,698 после поправки. Интервал сузился около 20% с 337,25 до 269,48 рублей. Оценка при этом сменила знак из-за перекоса spend_prev в 75,08 рублей в пользу теста(3259,08 против 3184,00): +24,22 -> -21,94 рублей . Исходный «плюс» в 24 ₽ объясняется перекосом групп, и CUPED его убрал, а эффекта акции по обороту не видно ни в плюс, ни в минус. p-value  после CUPED почти не изменилось: 0,7783 -> 0,7496

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [ ]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.

rng = np.random.default_rng(VARIANT)
x_t = test["spend"].to_numpy()
x_c = control["spend"].to_numpy()

diffs = np.empty(2000)
for i in range(2000):
    sample_t = rng.choice(x_t, size=len(x_t))   # сначала тест
    sample_c = rng.choice(x_c, size=len(x_c))   # потом контроль
    diffs[i] = sample_t.mean() - sample_c.mean()

lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"бутстреп: {lo:.2f} … {hi:.2f}  (нормальное приближение из задачи 3: {effect['lo']:.2f} … {effect['hi']:.2f})")
answer_9 = [round(float(lo), 2), round(float(hi), 2)]
answer_9

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [ ]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.

candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],
}
r2 = {name: mine["spend"].corr(col) ** 2 * 100 for name, col in candidates.items()}
for name, v in r2.items():
    print(f"{name:<24} ρ² = {v:.1f} %")

# для сравнения: покупки за время эксперимента (так брать нельзя)
print(f"{'purchases (в эксперименте)':<24} ρ² = {mine['spend'].corr(mine['purchases']) ** 2 * 100:.1f} %")

best = max(r2, key=r2.get)
answer_10 = [best, round(float(r2[best]), 1)]
answer_10

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.

feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]

# HypEx ждёт группу числом: 1 у теста, 0 у контроля. Копия, чтобы не менять mine.
mt = mine.copy()
mt["t"] = (mt["grp"] == "test").astype(int)

roles_m = {"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
           **{f: FeatureRole() for f in feats}}

res_match = Matching(n_neighbors=5).execute(
    Dataset(roles=dict(roles_m), data=mt, default_role=InfoRole()))
match_table = F(res_match.resume)
print(match_table.to_string())

ate = match_table.loc["ATE"]
answer_11 = [round(float(ate["Effect Size"]), 2),
             round(float(ate["CI Lower"]), 2),
             round(float(ate["CI Upper"]), 2)]
answer_11

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [ ]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽

gain = effect_cuped["diff"] * COMMISSION                     # комиссия с прироста оборота (CUPED)
ticket_cost = guards.loc["tickets", "diff"] * TICKET_COST    # расход: больше обращений = минус
churn_cost = guards.loc["churned", "diff"] * CHURN_COST      # расход: больше оттока = минус

total = gain - ticket_cost - churn_cost

print(f"комиссия с прироста оборота: {effect_cuped['diff']:.2f} × {COMMISSION} = {gain:.2f} ₽")
print(f"обращения: {guards.loc['tickets', 'diff']:.4f} × {TICKET_COST} = {ticket_cost:.2f} ₽ (вычитаем)")
print(f"отток: {guards.loc['churned', 'diff']:.4f} × {CHURN_COST} = {churn_cost:.2f} ₽ (вычитаем)")
print(f"итого на клиента за 4 недели: {total:.2f} ₽")
answer_12 = round(float(total), 2)
answer_12

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [ ]:
widths = {
    "обычный (задача 3)": effect["hi"] - effect["lo"],
    "CUPED (задача 7)": effect_cuped["hi"] - effect_cuped["lo"],
    "бутстреп (задача 9)": answer_9[1] - answer_9[0],
    "мэтчинг (задача 11)": answer_11[2] - answer_11[1],
}
for name, w in widths.items():
    print(f"{name:<22} ширина {w:.2f} ₽")
print("самый узкий:", min(widths, key=widths.get))

rho2_purchases = mine["spend"].corr(mine["purchases"]) ** 2 * 100
print(f"purchases: ρ² = {rho2_purchases:.1f} % (измерена во время эксперимента)")
print(f"снижение дисперсии у CUPED: {answer_7[1]} %")

print(f"итог акции: {answer_12:.2f} ₽ на клиента за 4 недели")
print(f"  из них отток: {-guards.loc['churned', 'diff'] * CHURN_COST:.2f} ₽, p оттока = {answer_5[1]}")
print(f"  без оттока итог был бы: {answer_12 + guards.loc['churned', 'diff'] * CHURN_COST:.2f} ₽")

#### ✍️ Ответ на вопрос 3

Из всех 4 вариантов(337,25 (обычный), 332,24 (бутстреп), 274,99 (мэтчинг), 269,48 (CUPED)) больше всего я доверяю интервалу с CUPED, так как он снижает дисперсию на 36,1 % за счёт связи с прошлым оборотом. Метчинг добавляет свой шум при случайном делении группы, а бутстрап близок к нормальному приближению из-за большой выборки. Purchases с ρ² = 71,6 % измерена уже во время эксперимента, поэтому CUPED вычел бы вместе с шумом и часть настоящего эффекта. Акция формально окупается (+2,33 рубля на клиента), но этот результат держится на незначимом оттоке. До полного расчета не хватает стоимости самой акции + расчёт только за один период, а эффект может затухать или копиться

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

In [ ]:
sigma_cuped = sigma * (1 - rho ** 2) ** 0.5
print("клиентов на группу, чтобы заметить 100 ₽:", n_per_group(sigma_cuped, 100))

#### ✍️ Отчёт продакту

**Что проверяли:** Акция «Кэшбэк 5 % на аптеку», 6 000 клиентов (3 003 в тесте, 2 997 в контроле), 4 недели, главная метрика оборот. Деление корректно: SRM p = 0,938, баланс по прошлому обороту p = 0,372. Ожидали роста оборота на 200 рублей.

**Что получилось:** 
Эффект на оборот: −21,94 ₽, интервал от −156,68 до 112,80 с CUPED(основная оценка по дизайну) содержит ноль. Без CUPED было +24,22 ₽ (от −144,40 до 192,85).То есть, рост не доказан.
Защитные метрики не ухудшились:
Отток: −0,41 п.п., p = 0,405.
Обращения: +0,003 на клиента, p = 0,663.
В деньгах: +2,33 ₽ на клиента за 4 недели, из них отток даёт 3,05 ₽, без оттока получилось бы −0,72 ₽.
После Холма значимых метрик нет (минимальное p после поправки 0,698).

**Какие остались вопросы:** Тест замечает эффект от 187 рублей, а минимальный необходимый рост =  100 рублей. Интервал его не исключает, поэтому нельзя сказать, что акция не работает. Джае 100 рублей роста дают всего 1,5 рубля на клиента, но это легко съедает небольшой рост оттока клиентов. Для окупаемости с запасом необходим рост в сотни рублей. В расчете не участвует кешбек, но мы допускаем, что его платит партнер. Если платит Прайм, то итог хуже. Также расчет сделан только за 4 недели. Верхняя граница 112,80 ниже ожидаемых 200 ₽, то есть роста, на который рассчитывали, акция не дала. 

**Что рекомендуем:** По правилу эксперимента необходимо его дорабатывать: нижняя граница -156,68, а верхняя 112,8, порог в 100 рублей находится внутри интервала. Нужно повторить эксперимент с около 10 500 клиентов на группу, либо пересмотреть условия акции, так как для окупаемости нужны сотни рублей роста.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [ ]:
# Исследование А: ваш код. Всё — от mine.
feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]
med = mine["sessions_prev"].median()

# 1. «Наблюдательные» данные: из теста только активные, из контроля все
obs = pd.concat([
    test[test["sessions_prev"] >= med],
    control,
]).copy()
obs["t"] = (obs["grp"] == "test").astype(int)
print(f"медиана sessions_prev: {med}, активных в тесте: {obs['t'].sum()}, контроль: {(obs['t'] == 0).sum()}")

# 2. Наивная разность
naive = diff_ci(obs.loc[obs["t"] == 1, "spend"], obs.loc[obs["t"] == 0, "spend"])
print(f"наивная разность: {naive['diff']:.2f} ₽ ({naive['lo']:.2f} … {naive['hi']:.2f})")

# 3. Мэтчинг, ATT — эффект на тех, кто акцию увидел
def att_matching(data, features):
    roles = {"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
             **{f: FeatureRole() for f in features}}
    res = Matching(n_neighbors=5).execute(Dataset(roles=dict(roles), data=data, default_role=InfoRole()))
    return F(res.resume).loc["ATT"]

att_full = att_matching(obs, feats)
print(f"мэтчинг ATT, все признаки: {att_full['Effect Size']:.2f} ₽ "
      f"({att_full['CI Lower']:.2f} … {att_full['CI Upper']:.2f})")

# 4. Эталон: активные из теста против активных из контроля (это даёт рандомизация)
bench = diff_ci(test.loc[test["sessions_prev"] >= med, "spend"],
                control.loc[control["sessions_prev"] >= med, "spend"])
print(f"эталон: {bench['diff']:.2f} ₽ ({bench['lo']:.2f} … {bench['hi']:.2f})")

# 5. Мэтчинг без признака, по которому шёл отбор
att_short = att_matching(obs, [f for f in feats if f != "sessions_prev"])
print(f"мэтчинг ATT без sessions_prev: {att_short['Effect Size']:.2f} ₽ "
      f"({att_short['CI Lower']:.2f} … {att_short['CI Upper']:.2f})")

# Сводка
summary_a = pd.DataFrame({
    "оценка": ["наивная", "мэтчинг, все признаки", "мэтчинг без sessions_prev", "эталон"],
    "эффект": [naive["diff"], att_full["Effect Size"], att_short["Effect Size"], bench["diff"]],
}).set_index("оценка")
summary_a["смещение от эталона"] = summary_a["эффект"] - bench["diff"]
print(summary_a.round(2).to_string())

# Почему мэтчинг без sessions_prev не вернулся к наивной оценке:
# насколько spend_prev «заменяет» выброшенный признак
corr_ss = mine["sessions_prev"].corr(mine["spend_prev"])
print(f"корреляция sessions_prev и spend_prev: {corr_ss:.3f}")

# Насколько различались активные и неактивные по прошлому обороту
active = mine["sessions_prev"] >= med
print(f"spend_prev: активные {mine.loc[active, 'spend_prev'].mean():.2f} ₽, "
      f"остальные {mine.loc[~active, 'spend_prev'].mean():.2f} ₽")

#### ✍️ Исследование А

Вопрос: насколько наивное сравнение завышает эффект, если акцию увидели только активные, и помогает ли мэтчинг

Как проверяли:

Наблюдательные данные: из теста клиенты с sessions_prev не ниже медианы в 15, это 1546 человек, а в контроле 2997 человек.
Сравнили наивную разность средних, мэтчинг ATT по шести признакам и мэтчинг без sessions_prev.
Эталон дает рандомизация, поэтому он считается правдой.

Что получилось:

| Оценка | Эффект, ₽ | Интервал, ₽ | Смещение от эталона, ₽ |
|---|---|---|---|
| наивная разность | 1 405,21 | (1 181,04; 1 629,37) | 1 403,99 |
| мэтчинг ATT, все признаки | 39,23 | (−131,67; 210,13) | 38,01 |
| мэтчинг ATT без sessions_prev | 115,59 | (−67,91; 299,10) | 114,37 |
| эталон (рандомизация) | 1,22 | (−273,69; 276,13) | 0 |

Активные до акции тратили 4 665,18 ₽, остальные 1 714,03 ₽. Корреляция sessions_prev и spend_prev 0,714.

Что это значит:

1405 рублей получены путем отсева выборки по медиане столбца sessions_prev, из теста взяли только активных, а они и до акции тратили больше (4 665,18 против 1 714,03 ₽).
Оценка метчинга совместима с эталоном,так как эталон 1,22 ₽ лежит внутри интервала мэтчинга (−131,67;210,13)
При метчинге на всех признаках и без sessions_prev мы убрали признак, который имел корреляцию с другим признаком spend_prev, поэтому оценка сместилась (39,23 против 115,59), но не вернулась к наивной(1405), так как spend_prev связан с сессиями(корреляция = 0,714) и частично заменил отброшенный признак.
Метчингу можно доверять, когда признак, по которому шёл отбор (или сильно связанный с ним), есть среди признаков мэтчинга


Чего расчет не доказывает:

Эталон сам по себе шумный (широкий интервал), поэтому 39 и 1 рубль по этим данным не различить.
Здесь отбор шёл по наблюдаемому признаку, и мы знали, по какому. В жизни отбор часто идёт по тому, чего нет в данных.
Это один прогон на одном варианте, и качество пар мы не проверяли.



### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [ ]:
# Исследование Б: ваш код. Всё — от mine.
rng = np.random.default_rng(VARIANT)

y_c = control["spend"]
x_c = control["spend_prev"]
effect_b = 0.03 * y_c.mean()                       # +3 % к среднему обороту контроля

theta_c = y_c.cov(x_c) / x_c.var()                 # θ по контрольной группе
y_c_cuped = (y_c - theta_c * (x_c - x_c.mean())).to_numpy()
y_c_raw = y_c.to_numpy()

sizes_b = [1000, 3000, 10000, 20000, 40000]
n_sim = 500

def sim_power(y, n):
    hits = 0
    for _ in range(n_sim):
        s = rng.choice(y, size=2 * n)
        a, b = s[:n] + effect_b, s[n:]
        hits += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
    return hits / n_sim

def formula_power(sigma_, n):
    z = stats.norm.ppf(0.975)
    return stats.norm.cdf(effect_b / (sigma_ * (2 / n) ** 0.5) - z)

rows = []
for n in sizes_b:
    rows.append({
        "n на группу": n,
        "симуляция": sim_power(y_c_raw, n),
        "формула": formula_power(y_c_raw.std(ddof=1), n),
        "симуляция CUPED": sim_power(y_c_cuped, n),
        "формула CUPED": formula_power(y_c_cuped.std(ddof=1), n),
    })
power_b = pd.DataFrame(rows).set_index("n на группу")

print(f"эффект +3 %: {effect_b:.2f} ₽, θ по контролю: {theta_c:.3f}")
print(power_b.round(3).to_string())
print("n для 80 % по формуле: без CUPED", n_per_group(y_c_raw.std(ddof=1), effect_b),
      "· с CUPED", n_per_group(y_c_cuped.std(ddof=1), effect_b))
print("в нашем эксперименте на группу:", len(test), "и", len(control))
print(f"погрешность доли при 500 повторах: ±{(0.5 * 0.5 / 500) ** 0.5 * 100:.1f} п.п.")

#### ✍️ Исследование Б

Вопрос: сколько клиентов на группу нужно, чтобы заметить рост оборота на 3 %, с CUPED и без, и не врёт ли формула мощности, когда у оборота тяжёлый хвост.

как проверяли: 
Взяли оборот контроля (2 997 клиентов, эффекта там нет), эффект +3 % = 99,8 рублей.
Для n = 1 000, 3 000, 10 000, 20 000, 40 000 по 500 раз: выбор с возвращением 2n значений, к первой половине прибавили эффект, сравнили t-критерием Уэлча. Мощность - доля отвержений при α = 0,05.
То же для оборота после CUPED (θ по контролю = 0,617, ковариата spend_prev), посчитанного заранее для каждого клиента.
Сравнили с формулой мощности из С7 (★3).

Что получилось:

| n на группу | симуляция | формула | симуляция CUPED | формула CUPED |
|---|---|---|---|---|
| 1 000 | 0,110 | 0,097 | 0,090 | 0,130 |
| 3 000 | 0,204 | 0,208 | 0,268 | 0,303 |
| 10 000 | 0,582 | 0,552 | 0,750 | 0,751 |
| 20 000 | 0,860 | 0,840 | 0,960 | 0,962 |
| 40 000 | 0,992 | 0,987 | 1,000 | 1,000 |

n для 80 % по формуле: без CUPED 17961 · с CUPED 11275; в нашем эксперименте на группу: 3003 и 2997.


Что это значит:

По симуляции 80 % достигаются между 10 000 и 20 000 (без CUPED 0,582 → 0,860, с CUPED 0,750 → 0,960), по формуле 17 961 и 11 275, то есть экономия 6 694 человека на группу (около 37 %). Так как CUPED снижает дисперсию в 36,1% из задачи 7, то это снижает необходимую выборку (по формуле с лекций выборка пропорционально зависит от дисперсии). 
Мощность на выборке около 3000 человек составляет  0,204. Мощность около 20 % (с CUPED около 27 %) означает, что даже если бы акция реально давала +3 %, эксперимент такого размера заметил бы это лишь примерно в одном случае из 4-5. Поэтому незначимый результат задач 3 и 7 ожидаем и не говорит, что эффекта нет. 
Симуляция расходится с формулой на малых выборках в 1000 и 3000 человек, так как при малых n среднее из-за хвоста ещё не успело стать нормальным, а формула считает его нормальным. На больших выборках расхождение вызвано шумом симуляции. С CUPED по формуле мощность получается больше симуляции, поэтому формула завышает мощность, то есть, обещает больше шансов заметить эффект,  чем есть в действительности. 

Чего расчет не доказывает:

Погрешность симуляции при 500 повторах (около ±2,2 п.п.).
Хвост представлен только теми крупными тратами, что есть в наших 2997 клиентах.
Эффект прибавлялся всем одинаково, а в жизни он может быть разным.
θ оценена на тех же данных.